In [162]:
import pickle
import numpy as np
from pathlib import Path
from tqdm import tqdm
from vebir.pca import loo_pca, malinowski_ind
from vebir.ebs import EBS
from vebir.veb import VEB
from vebir.metrics_utils import compute_compound_cors, compute_method_cors
import pandas as pd 
import matplotlib.pyplot as plt

In [163]:
REPO_ROOT = Path.cwd().resolve().parent
CORRECTIONS_DIR = REPO_ROOT / "data" / "corrections" / "teflon"
GT_DIR = REPO_ROOT / "data" / "spectrabase" / "teflon"
LAB_DIR = REPO_ROOT / "data" / "raw" / "teflon" / "laboratory_samples"
PREPROC_DIR = REPO_ROOT / "data" / "preprocessed" / "teflon"

with open(PREPROC_DIR / "blanks_dict.pkl", "rb") as f:
    blanks_dict = pickle.load(f)

with open(GT_DIR / "ref_dict.pkl", 'rb') as file:
        ref_dict = pickle.load(file) 

del ref_dict["CO2"]

Z = np.array(blanks_dict["2011"])
    
wn = np.sort(pd.read_csv(LAB_DIR / "zerofilling.txt", header=None).iloc[:, 0])
compounds = ['12-Tricosanone', 'Ammonium sulfate', 'Malonic Acid', 'Suberic Acid', 'D-Glucose', 'fructose', 'levoglucosan']

In [164]:
chat_ind = malinowski_ind(Z)[0]
chat_loocv = loo_pca(Z)[1]
print(f"IND: {chat_ind}, LOOCV-OSE: {chat_loocv}")

ncomp_grid = 1 + np.arange(54)
tau_grid = 0.1 * np.power(2.0, np.arange(-2, 3))
a, b = np.meshgrid(ncomp_grid, tau_grid)
grid = np.column_stack([a.ravel(), b.ravel()])

with open(CORRECTIONS_DIR / "ebs_als_dict_V.pkl", "rb") as f:
   ebs_als_dict_V = pickle.load(f)
   
with open(CORRECTIONS_DIR / "ebs_als_dict_W.pkl", "rb") as f:
   ebs_als_dict_W = pickle.load(f)
   
with open(CORRECTIONS_DIR / "ebs_pb_dict_V.pkl", "rb") as f:
   ebs_pb_dict_V = pickle.load(f)

with open(CORRECTIONS_DIR / "ebs_pb_dict_W.pkl", "rb") as f:
   ebs_pb_dict_W = pickle.load(f)

IND: 53, LOOCV-OSE: 47


In [172]:
df = {}

corrections_dict = {}
key =f"ncomp,tau = {chat_ind},{0.10}"
for comp in compounds:
    corrections_dict[comp] = ebs_als_dict_V[comp][key]

df["EBS-ALS-IND"] = compute_method_cors(corrections_dict,ref_dict,wn)

corrections_dict = {}
key =f"ncomp,tau = {chat_loocv},{0.10}"
for comp in compounds:
    corrections_dict[comp] = ebs_als_dict_W[comp][key]

df["EBS-ALS-LOOCV"] = compute_method_cors(corrections_dict,ref_dict,wn)

corrections_dict = {}
key =f"ncomp,tau = {chat_ind},{0.10}"
for comp in compounds:
    corrections_dict[comp] = ebs_pb_dict_V[comp][key]

df["EBS-PB-IND"] = compute_method_cors(corrections_dict,ref_dict,wn)

corrections_dict = {}
key =f"ncomp,tau = {chat_loocv},{0.10}"
for comp in compounds:
    corrections_dict[comp] = ebs_pb_dict_W[comp][key]

df["EBS-PB-LOOCV"] = compute_method_cors(corrections_dict,ref_dict,wn)

for entry in df:
    for key in df[entry]:
        df[entry][key] = f"{df[entry][key]["mean"]:.2f} pm {df[entry][key]["std_err"]:.2f}"

df = pd.DataFrame(df).T
print(df)

              12-Tricosanone Ammonium sulfate   Malonic Acid   Suberic Acid  \
EBS-ALS-IND    64.67 pm 0.14    63.06 pm 0.40  62.20 pm 0.53  58.79 pm 0.29   
EBS-ALS-LOOCV  66.77 pm 0.10    63.49 pm 0.39  63.88 pm 0.40  61.11 pm 0.25   
EBS-PB-IND     87.00 pm 0.27    77.88 pm 0.27  69.26 pm 0.74  66.72 pm 0.75   
EBS-PB-LOOCV   87.07 pm 0.27    81.25 pm 0.21  69.76 pm 0.73  67.36 pm 0.54   

                   D-Glucose       fructose   levoglucosan            all  
EBS-ALS-IND    38.65 pm 0.56  50.14 pm 0.23  48.57 pm 0.46  57.60 pm 0.57  
EBS-ALS-LOOCV  39.23 pm 0.64  53.53 pm 0.19  50.93 pm 0.59  59.38 pm 0.55  
EBS-PB-IND     68.65 pm 1.94  83.01 pm 0.89  78.80 pm 1.58  78.01 pm 0.61  
EBS-PB-LOOCV   68.69 pm 1.95  84.24 pm 0.83  81.72 pm 1.76  79.73 pm 0.63  


In [173]:
df.drop(index=["EBS-ALS-IND", "EBS-PB-IND"], inplace=True)
df.rename(index={"EBS-ALS-LOOCV":"EBS-ALS" ,"EBS-PB-LOOCV":"EBS-PB"},inplace=True)

In [174]:
df_new = {}

max_corr = 0
best_cors_dict = None

for j in range(grid.shape[0]):
    corrections_dict = {}
    params = f"ncomp,tau = {int(grid[j, 0])},{grid[j, 1]}"
    for comp in ref_dict:
        corrections_dict[comp] = ebs_als_dict_W[comp][params]
    
    cors_dict =  compute_method_cors(corrections_dict, ref_dict, wn)   
    new_corr = cors_dict["all"]["mean"]
    
    if new_corr > max_corr:
        max_corr = new_corr
        best_params = params
        best_cors_dict = cors_dict 

print(f"EBS-ALS best params: {best_params}")
df_new["EBS-ALS*"] = best_cors_dict

max_corr = 0
best_cors_dict = None

for j in range(grid.shape[0]):
    corrections_dict = {}
    params = f"ncomp,tau = {int(grid[j, 0])},{grid[j, 1]}"
    for comp in ref_dict:
        corrections_dict[comp] = ebs_pb_dict_W[comp][params]
    
    cors_dict =  compute_method_cors(corrections_dict, ref_dict, wn)   
    new_corr = cors_dict["all"]["mean"]
    
    if new_corr > max_corr:
        max_corr = new_corr
        best_params = params
        best_cors_dict = cors_dict 

print(f"EBS-ALS best params: {best_params}")
df_new["EBS-PB*"] = best_cors_dict

EBS-ALS best params: ncomp,tau = 13,0.025
EBS-ALS best params: ncomp,tau = 14,0.2


In [175]:
for entry in df_new:
    for key in df_new[entry]:
        df_new[entry][key] = f"{df_new[entry][key]['mean']:.2f} pm {df_new[entry][key]['std_err']:.2f}"

In [176]:
df = pd.concat([df,pd.DataFrame(df_new).T])
df = df.reindex(["EBS-ALS","EBS-ALS*", "EBS-PB","EBS-PB*"])